In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import geopandas as gpd

from shapely.geometry import Point

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy.spatial import cKDTree

from sklearn.neighbors import KernelDensity

import rasterio
from rasterio.transform import from_origin

from pathlib import Path

In [3]:
INPUT_CSV = "/run/media/vincent/Extreme Pro/FaithAhiono/Species occurrence Africa/Species_occurrence/Species_occurrence/Butterfly_Moth.shp"

OUTPUT_GPKG = "clean_occurrences.gpkg"
OUTPUT_DENSITY = "sampling_density.tif"

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

DPI = 300

AFRICA_EQUAL_AREA = "EPSG:102022"  # Africa Albers Equal Area
WGS84 = "EPSG:4326"


In [4]:
df = gpd.read_file(INPUT_CSV)

print(df.shape)
df.head()

(108843, 5)


,latitude,longitude,scientific,common_nam,geometry
0,-12.385247,37.237427,Hemiolaus caeculus,Azure Hairstreak,POINT (37.23743 -12.38525)
1,-15.783571,35.006272,Junonia hierta cebrene,African Yellow Pansy,POINT (35.00627 -15.78357)
2,-15.364770,35.304866,Junonia terea,Soldier Pansy,POINT (35.30487 -15.36477)
3,-15.366912,35.299705,Tuxentius melaena,Black Pie,POINT (35.29971 -15.36691)
4,-15.327222,35.320833,Euchrysops malathana,Common Smoky Blue,POINT (35.32083 -15.32722)


In [5]:
df.columns = [c.lower().strip() for c in df.columns]

required = ["latitude", "longitude"]

for col in required:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")

df = df.rename(
    columns={
        "decimallatitude": "latitude",
        "decimallongitude": "longitude",
        "coordinateuncertaintyinmeters": "uncertainty_m"
    }
)

print(df.shape)


(108843, 5)


In [6]:
n_before = len(df)

df = df[
    (df["latitude"] >= -90) &
    (df["latitude"] <= 90) &
    (df["longitude"] >= -180) &
    (df["longitude"] <= 180)
]

df = df[
    ~(
        (df["latitude"] == 0) &
        (df["longitude"] == 0)
    )
]

print(f"Removed {n_before-len(df):,} invalid records")

Removed 0 invalid records


In [7]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(
        df.longitude,
        df.latitude
    ),
    crs=WGS84
)

gdf.head()

,latitude,longitude,scientific,common_nam,geometry
0,-12.385247,37.237427,Hemiolaus caeculus,Azure Hairstreak,POINT (37.23743 -12.38525)
1,-15.783571,35.006272,Junonia hierta cebrene,African Yellow Pansy,POINT (35.00627 -15.78357)
2,-15.364770,35.304866,Junonia terea,Soldier Pansy,POINT (35.30487 -15.36477)
3,-15.366912,35.299705,Tuxentius melaena,Black Pie,POINT (35.29971 -15.36691)
4,-15.327222,35.320833,Euchrysops malathana,Common Smoky Blue,POINT (35.32083 -15.32722)


In [8]:
world = gpd.read_file(
    gpd.datasets.get_path("naturalearth_lowres")
)

africa = world[
    world["continent"] == "Africa"
]

africa = africa.to_crs(WGS84)

africa.plot(figsize=(8,8))
plt.show()

AttributeError: The geopandas.dataset has been deprecated and was removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.

In [ ]:
n_before = len(gdf)

africa_union = africa.union_all()

gdf = gdf[
    gdf.within(africa_union)
]

print(f"Removed {n_before-len(gdf):,} ocean points")

In [ ]:
if "uncertainty_m" in gdf.columns:
    
    n_before = len(gdf)

    gdf = gdf[
        (
            gdf["uncertainty_m"].isna()
        ) |
        (
            gdf["uncertainty_m"] <= 5000
        )
    ]

    print(
        f"Removed {n_before-len(gdf):,} uncertain records"
    )

In [ ]:
countries = africa[
    ["name","geometry"]
]

gdf = gpd.sjoin(
    gdf,
    countries,
    predicate="within",
    how="left"
)

gdf = gdf.rename(
    columns={"name":"country"}
)

gdf.head()

In [ ]:
country_counts = (
    gdf["country"]
    .value_counts()
    .reset_index()
)

country_counts.columns = [
    "country",
    "n_records"
]

country_counts.head()

In [ ]:
gdf_proj = gdf.to_crs(AFRICA_EQUAL_AREA)

coords = np.array([
    (
        geom.x,
        geom.y
    )
    for geom in gdf_proj.geometry
])

tree = cKDTree(coords)

distances, idx = tree.query(
    coords,
    k=2
)

nn_distance = distances[:,1]

gdf_proj["nn_distance_m"] = nn_distance

gdf_proj["nn_distance_km"] = (
    nn_distance / 1000
)

print(
    gdf_proj["nn_distance_km"].describe()
)

In [ ]:
xmin, ymin, xmax, ymax = (
    gdf_proj.total_bounds
)

resolution = 25000

xgrid = np.arange(
    xmin,
    xmax,
    resolution
)

ygrid = np.arange(
    ymin,
    ymax,
    resolution
)

xx, yy = np.meshgrid(
    xgrid,
    ygrid
)

grid_points = np.vstack(
    [xx.ravel(), yy.ravel()]
).T

bandwidth = 100000

kde = KernelDensity(
    bandwidth=bandwidth,
    kernel="gaussian"
)

kde.fit(coords)

density = np.exp(
    kde.score_samples(
        grid_points
    )
)

density = density.reshape(
    yy.shape
)

In [ ]:
transform = from_origin(
    xmin,
    ymax,
    resolution,
    resolution
)

with rasterio.open(
    OUTPUT_DENSITY,
    "w",
    driver="GTiff",
    height=density.shape[0],
    width=density.shape[1],
    count=1,
    dtype="float32",
    crs=AFRICA_EQUAL_AREA,
    transform=transform,
    compress="lzw"
) as dst:

    dst.write(
        density.astype("float32"),
        1
    )

print(OUTPUT_DENSITY)

In [ ]:
fig, ax = plt.subplots(
    figsize=(12,10)
)

africa.plot(
    ax=ax,
    color="lightgrey",
    edgecolor="black"
)

gdf.plot(
    ax=ax,
    markersize=2,
    alpha=0.5
)

ax.set_title(
    "Occurrence Records Across Africa",
    fontsize=18
)

plt.tight_layout()

plt.savefig(
    FIG_DIR/"Figure1_Africa_Map.png",
    dpi=DPI
)

plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(12,10)
)

im = ax.imshow(
    density,
    cmap="viridis",
    origin="lower"
)

plt.colorbar(
    im,
    ax=ax,
    label="Sampling Density"
)

ax.set_title(
    "Sampling Density Surface",
    fontsize=18
)

plt.tight_layout()

plt.savefig(
    FIG_DIR/"Figure2_Density.png",
    dpi=DPI
)

plt.show()

In [ ]:
top = (
    country_counts
    .sort_values(
        "n_records",
        ascending=False
    )
    .head(25)
)

fig, ax = plt.subplots(
    figsize=(12,8)
)

sns.barplot(
    data=top,
    y="country",
    x="n_records",
    ax=ax
)

ax.set_title(
    "Country-Level Occurrence Counts",
    fontsize=18
)

plt.tight_layout()

plt.savefig(
    FIG_DIR/"Figure3_Country_Counts.png",
    dpi=DPI
)

plt.show()

In [ ]:
if "biome" in gdf.columns:
    
    biome_counts = (
        gdf["biome"]
        .value_counts()
    )

    fig, ax = plt.subplots(
        figsize=(10,6)
    )

    biome_counts.plot(
        kind="bar",
        ax=ax
    )

    ax.set_title(
        "Biome-Level Occurrence Counts",
        fontsize=18
    )

    plt.tight_layout()

    plt.savefig(
        FIG_DIR/"Figure4_Biomes.png",
        dpi=DPI
    )

    plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(10,6)
)

sns.histplot(
    gdf["latitude"],
    bins=50,
    kde=True,
    ax=ax
)

ax.set_title(
    "Latitudinal Distribution",
    fontsize=18
)

ax.set_xlabel(
    "Latitude"
)

plt.tight_layout()

plt.savefig(
    FIG_DIR/"Figure5_Latitude.png",
    dpi=DPI
)

plt.show()

In [ ]:
gdf.to_file(
    OUTPUT_GPKG,
    driver="GPKG"
)

print(f"Saved: {OUTPUT_GPKG}")
print(f"Saved: {OUTPUT_DENSITY}")